### Baseline Modeling

We'll train a couple of simple models using their default settings and evaluate their performance on this highly imbalanced dataset.

At this stage, we're not using SMOTE, class weights, or hyperparameter tuning. The idea is simply to see how well standard models perform and create a reference point for later experiments.

**Models:**
- Logistic Regression
- Random Forest

**Metrics tracked:**
- PR-AUC (main)
- Recall, Precision, F1, ROC-AUC

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [ ]:
data = pd.read_csv('../data/creditcard.csv')

In [ ]:
X = data.drop(columns=["Class"])
y = data["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train fraud rate:", y_train.mean())
print("Test fraud rate:", y_test.mean())

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), ['Amount', 'Time'])
    ],
    remainder='passthrough'
)

lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])


In [ ]:
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = ['average_precision', 'precision', 'recall', 'f1', 'roc_auc']

cv_results_lr = cross_validate(
    estimator = lr_pipeline,
    X = X_train,
    y = y_train,
    cv = skf,
    scoring = scoring,
    n_jobs = 4
)

cv_results_rf = cross_validate(
    estimator = rf_pipeline,
    X = X_train,
    y = y_train,
    cv = skf,
    scoring = scoring,
    n_jobs = 4
)

In [ ]:
print("LR keys:", cv_results_lr.keys())
print("LR PR-AUC fold scores:", cv_results_lr['test_average_precision'])

In [ ]:
for metric in ['test_average_precision', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']:
    print(f"LR {metric} mean: {cv_results_lr[metric].mean():.3f}, std: {cv_results_lr[metric].std():.3f}")

In [ ]:
for metric in ['test_average_precision', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']:
    print(f"RF {metric} mean: {cv_results_rf[metric].mean():.3f}, std: {cv_results_rf[metric].std():.3f}")

### Results
Logistic Regression
1. PR-AUC mean: 0.762, std: 0.042
2. Precision mean: 0.875, std: 0.037
3. Recall mean: 0.642, std: 0.029
4. F1 mean: 0.740, std: 0.014
5. ROC mean: 0.978, std: 0.011

Random Forest
1. PR-AUC mean: 0.840, std: 0.032
2. Precision mean: 0.940, std: 0.036
3. Recall mean: 0.766, std: 0.021
4. F1 mean: 0.844, std: 0.015
5. ROC mean: 0.949, std: 0.016



### Key observations

1. **Random Forest does better job at 4 out of 5 metrics**
2. **Important note on ROC-AUC vs PR-AUC**: LR has higher ROC-AUC (0.978) than RF (0.949), 
but LOWER PR-AUC (0.762 vs 0.840). This confirms ROC-AUC can be misleading on imbalanced data
3. **High precision, low recall**:
   - **LR**: precision 0.875, recall 0.642
   - **RF**: precision 0.940, recall 0.766
   
    Both models alarm accurately (87.5% for LR, 94% for RF), but miss many frauds (35.8% for LR, 23.4% for RF). They are conservative, which may not align with the business need.



### Next steps
In notebook 03 we'll add imbalance handling techniques (class_weight, SMOTE) to push recall higher, even with the cost of precision.